# Phase 4 · Architecture 4: LegalBlock-GQA — Court judgment prediction

In [1]:
%pip -q install "tokenizers==0.21.4" "huggingface_hub==0.34.4" "cloudpickle==3.1.1" scikit-learn


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.34.4 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.34.4 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.34.4 which is incompatible.
transformers 5.16.1 requires tokenizers<0.24.0,>=0.23.1, but you have tokenizers 0.21.4 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import shutil

SOURCE = Path('/kaggle/input/datasets/raaz74/a4-p4-resume2/phase4_a4_scratch_train_only')
OUT = Path('/kaggle/working/phase4_a4_scratch_train_only')

shutil.copytree(SOURCE, OUT, dirs_exist_ok=True)

PosixPath('/kaggle/working/phase4_a4_scratch_train_only')

In [3]:
import os, sys, json, math, time, random, gc, zipfile, subprocess, cloudpickle, hashlib, shutil
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfApi
from sklearn.metrics import f1_score, classification_report

BASE = Path('/kaggle/input/datasets/raaz74/a4-p4')
HF_TOKEN = os.environ['HF_TOKEN']
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS'] = '2'
OUT.mkdir(parents=True, exist_ok=True)
REPO = 'Exploration-Lab/CS-781-Capstone'
C = json.loads((BASE/'config.json').read_text())
R = dict(base=str(BASE), out=str(OUT), seed=781, length=512, stride=384,
         chunks=6, chunk_batch=2, micro=1, accum=8, rank=16,
         eval_steps=200, patience=5, classes=2)
METHODS = {
    'full': dict(epochs=6, lr=2e-5, coverage='uniform'),
    'partial': dict(epochs=8, lr=5e-5, coverage='head_tail'),
    'lora': dict(epochs=10, lr=2e-4, coverage='uniform'),
    'lorma': dict(epochs=10, lr=2e-4, coverage='uniform'),
    'ia3': dict(epochs=10, lr=5e-4, coverage='all'),
    'distill': dict(epochs=8, lr=4e-5, coverage='uniform'),
}
random.seed(R['seed'])
np.random.seed(R['seed'])
torch.manual_seed(R['seed'])


## Training and development data

In [4]:
revision = HfApi(token=HF_TOKEN).dataset_info(REPO).sha
files = {'train':'Train/train.json', 'dev':'Train/dev.json'}
tok = Tokenizer.from_file(str(BASE/'tokenizer.json'))
R['pad'], R['eos'] = tok.token_to_id('<pad>'), tok.token_to_id('<eos>')

def read_docs(raw, labelled):
    if isinstance(raw, dict):
        raw = [raw] if 'id' in raw and 'text' in raw else [dict(v, id=k) for k,v in raw.items()]
    docs = []
    for row in raw:
        text = row['text']
        if isinstance(text, list):
            text = '\n'.join(text)
        y = -100
        if labelled:
            y = row['label'] if 'label' in row else row['labels']
            if isinstance(y, list) and len(y)==1: y = y[0]
            if y not in [0,1,'0','1']:
                raise ValueError('Expected a binary label for '+str(row['id']))
            y = int(y)
        docs.append(dict(id=row['id'], text=text, y=y))
    return docs

def cache_docs(split, docs):
    rows, offset = [], 0
    with (OUT/f'{split}_tokens.bin').open('wb') as f:
        for doc in docs:
            ids = np.asarray(tok.encode(doc['text'],add_special_tokens=False).ids, dtype=np.int32)
            ids.tofile(f)
            row = dict(id=doc['id'], offset=offset, size=len(ids))
            if split!='test': row['y'] = doc['y']
            rows.append(row)
            offset += len(ids)
        if offset==0: np.array([R['eos']],dtype=np.int32).tofile(f)
    print(f'{split}: {len(rows):,} cases | {offset:,} BPE tokens')
    return rows

DATA = {}
for split,name in files.items():
    path = hf_hub_download(REPO, 'Phase_4_CJPE/'+name, repo_type='dataset',
                           revision=revision, token=HF_TOKEN)
    docs = read_docs(json.loads(Path(path).read_text()), labelled=True)
    DATA[split] = cache_docs(split,docs)
    del docs
with (OUT/'train_dev.pkl').open('wb') as f: cloudpickle.dump(DATA,f)
identity = dict(config=C, settings=R.copy(), methods=METHODS, revision=revision,
                data_policy='train_only_separate_inference_v2',
                adapter_implementation='handwritten_pytorch_v1', architecture=4,
                tokenizer_sha256=hashlib.sha256((BASE/'tokenizer.json').read_bytes()).hexdigest())
with (BASE/'best.pt').open('rb') as f:
    digest = hashlib.sha256()
    for block in iter(lambda:f.read(8*1024*1024),b''): digest.update(block)
identity['base_sha256'] = digest.hexdigest()
R['run_id'] = hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()
manifest = dict(**identity, run_id=R['run_id'], repo=REPO, files=files)
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
shutil.copyfile(BASE/'tokenizer.json',OUT/'tokenizer.json')


Phase_4_CJPE/Train/train.json:   0%|          | 0.00/737M [00:00<?, ?B/s]

train: 37,387 cases | 149,567,517 BPE tokens


Phase_4_CJPE/Train/dev.json:   0%|          | 0.00/45.0M [00:00<?, ?B/s]

dev: 1,988 cases | 10,027,740 BPE tokens


PosixPath('/kaggle/working/phase4_a4_scratch_train_only/tokenizer.json')

## LegalBlock-GQA backbone and classification head

In [5]:
class RMSNorm(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return (x.float() * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + 1e-6)).to(x.dtype) * self.weight

class Attention(nn.Module):
    def __init__(self, c, global_attn):
        super().__init__()
        self.h, self.kv, self.d = c['heads'], c['kv_heads'], c['dim'] // c['heads']
        self.window, self.global_attn = c['window'], global_attn
        self.qkv = nn.Linear(c['dim'], (self.h + 2*self.kv)*self.d, bias=False)
        self.out = nn.Linear(c['dim'], c['dim'], bias=False)
        self.register_buffer('freq', 1 / (10000 ** (torch.arange(0, self.d, 2).float()/self.d)), persistent=False)
    def rope(self, x, pos):
        angle = (torch.arange(pos, pos+x.size(2), device=x.device).float()[:, None] * self.freq).to(x.dtype)
        a, b = x[..., 0::2], x[..., 1::2]
        return torch.stack((a*angle.cos()-b*angle.sin(), a*angle.sin()+b*angle.cos()), -1).flatten(-2)
    def forward(self, x, cache=None, pos=0, use_cache=False):
        b, t, _ = x.shape
        q, k, v = self.qkv(x).split([self.h*self.d, self.kv*self.d, self.kv*self.d], -1)
        q = self.rope(q.view(b,t,self.h,self.d).transpose(1,2), pos)
        k = self.rope(k.view(b,t,self.kv,self.d).transpose(1,2), pos)
        v = v.view(b,t,self.kv,self.d).transpose(1,2)
        if cache is not None and (self.global_attn or pos % self.window != 0):
            k, v = torch.cat((cache[0], k), 2), torch.cat((cache[1], v), 2)
        saved = None
        if use_cache:
            start = 0 if self.global_attn else ((pos+t-1)//self.window)*self.window - pos
            start = max(0, start) if cache is None else 0
            saved = (k[:,:,start:], v[:,:,start:])
        k, v = k.repeat_interleave(self.h//self.kv, 1), v.repeat_interleave(self.h//self.kv, 1)
        if t == 1:
            y = F.scaled_dot_product_attention(q, k, v)
        elif self.global_attn:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            w = self.window
            pad = (-t) % w
            q, k, v = [F.pad(z, (0,0,0,pad)).reshape(b,self.h,-1,w,self.d)
                       .permute(0,2,1,3,4).reshape(-1,self.h,w,self.d) for z in (q,k,v)]
            y = F.scaled_dot_product_attention(q,k,v,is_causal=True)
            y = y.reshape(b,-1,self.h,w,self.d).permute(0,2,1,3,4).reshape(b,self.h,-1,self.d)[:,:,:t]
        return self.out(y.transpose(1,2).reshape(b,t,-1)), saved

class Block(nn.Module):
    def __init__(self, c, i):
        super().__init__()
        self.n1, self.n2 = RMSNorm(c['dim']), RMSNorm(c['dim'])
        self.attn = Attention(c, (i+1) % c['global_every'] == 0)
        self.up = nn.Linear(c['dim'], 2*c['ff'], bias=False)
        self.down = nn.Linear(c['ff'], c['dim'], bias=False)
    def forward(self, x, cache=None, pos=0, use_cache=False):
        y, cache = self.attn(self.n1(x), cache, pos, use_cache)
        x = x + y
        a, b = self.up(self.n2(x)).chunk(2, -1)
        return x + self.down(F.silu(a)*b), cache

class LegalLM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.emb = nn.Embedding(c['vocab'], c['dim'])
        self.blocks = nn.ModuleList([Block(c, i) for i in range(c['layers'])])
        self.norm = RMSNorm(c['dim'])
        self.apply(self.init_weights)
        for block in self.blocks:
            nn.init.normal_(block.attn.out.weight, std=.02/math.sqrt(2*c['layers']))
            nn.init.normal_(block.down.weight, std=.02/math.sqrt(2*c['layers']))
    def init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=.02)


In [6]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank=16, alpha=32, dropout=.05):
        super().__init__()
        self.base = base.requires_grad_(False)
        self.a = nn.Parameter(torch.empty(rank,base.in_features))
        self.b = nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.a,a=math.sqrt(5))
        self.scale, self.drop = alpha/rank, nn.Dropout(dropout)
    def forward(self, x):
        update = F.linear(F.linear(self.drop(x),self.a),self.b)
        return self.base(x) + self.scale*update

class LoRMALinear(nn.Module):
    def __init__(self, base, rank=16, alpha=32, dropout=.05):
        super().__init__()
        self.base = base.requires_grad_(False)
        self.a = nn.Parameter(torch.empty(rank,base.out_features))
        self.b = nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.a,a=math.sqrt(5))
        self.scale, self.drop = alpha/rank, nn.Dropout(dropout)
    def forward(self, x):
        y = F.linear(x,self.base.weight)
        update = F.linear(F.linear(self.drop(y),self.a),self.b)
        y = y + self.scale*update
        return y if self.base.bias is None else y+self.base.bias

class IA3Linear(nn.Module):
    def __init__(self, base, feedforward=False):
        super().__init__()
        self.base, self.feedforward = base.requires_grad_(False), feedforward
        size = base.in_features if feedforward else base.out_features
        self.scale = nn.Parameter(torch.ones(size))
    def forward(self, x):
        if self.feedforward: return self.base(x*self.scale.to(x.dtype))
        y = self.base(x)
        return y*self.scale.to(y.dtype)

class SplitQKV(nn.Module):
    def __init__(self, base, c):
        super().__init__()
        d = c['dim']
        kv = c['kv_heads']*(d//c['heads'])
        self.q_proj, self.k_proj, self.v_proj = [nn.Linear(d,k,bias=False) for k in [d,kv,kv]]
        with torch.no_grad():
            for layer,w in zip([self.q_proj,self.k_proj,self.v_proj],base.weight.split([d,kv,kv],0)):
                layer.weight.copy_(w)
    def forward(self, x):
        return torch.cat((self.q_proj(x),self.k_proj(x),self.v_proj(x)),-1)

class CaseModel(nn.Module):
    def __init__(self, lm, method, r, coverage):
        super().__init__()
        self.lm, self.r, self.coverage = lm, r, coverage
        d = lm.c['dim']
        self.chunk_head = nn.Sequential(nn.Linear(2*d+1,256),nn.GELU(),nn.Dropout(.15))
        self.pool = nn.Linear(256,1)
        self.head = nn.Sequential(nn.Dropout(.15),nn.Linear(256,2))
    def encode(self, x, lengths):
        h = self.lm.emb(x)
        for block in self.lm.blocks:
            h,_ = checkpoint(block,h,use_reentrant=False) if self.training else block(h)
        h = self.lm.norm(h)
        mask = torch.arange(x.size(1),device=x.device)[None]<lengths[:,None]
        mean = (h.float()*mask[:,:,None]).sum(1)/lengths[:,None]
        last = h[torch.arange(x.size(0),device=x.device),lengths-1].float()
        return torch.cat((mean,last),-1)
    def forward(self, tokens):
        starts = chunk_starts(len(tokens),self.r,self.coverage)
        device = self.lm.emb.weight.device
        reps = []
        for at in range(0,len(starts),self.r['chunk_batch']):
            group = starts[at:at+self.r['chunk_batch']]
            parts = [np.append(tokens[s:s+self.r['length']-1],self.r['eos']) for s in group]
            lengths = torch.tensor([len(v) for v in parts],device=device)
            width = math.ceil(max(map(len,parts))/8)*8
            x = torch.full((len(parts),width),self.r['pad'],dtype=torch.long,device=device)
            for i,v in enumerate(parts): x[i,:len(v)] = torch.as_tensor(v,dtype=torch.long,device=device)
            h = checkpoint(self.encode,x,lengths,use_reentrant=False) if self.training else self.encode(x,lengths)
            reps.append(h)
        pos = torch.tensor(starts,dtype=torch.float32,device=device)[:,None]/max(1,len(tokens)-1)
        h = self.chunk_head(torch.cat((torch.cat(reps),pos),-1))
        weights = self.pool(h).float().softmax(0)
        return self.head((h*weights).sum(0,keepdim=True))

def chunk_starts(size, r, coverage):
    last = max(0,size-(r['length']-1))
    starts = list(range(0,last+1,r['stride']))
    if starts[-1]!=last: starts.append(last)
    if coverage=='head_tail': return sorted(set([0,last]))
    if coverage=='uniform' and len(starts)>r['chunks']:
        starts = [starts[i] for i in np.linspace(0,len(starts)-1,r['chunks'],dtype=int)]
    return starts

def make_model(method, c, r, methods):
    saved = torch.load(Path(r['base'])/'best.pt',map_location='cpu',weights_only=False)
    config = saved.get('config',c).copy()
    state = saved['model']
    chosen = list(range(config['layers']))
    if method=='distill':
        chosen = np.linspace(0,config['layers']-1,8,dtype=int).tolist()
        config['layers'] = len(chosen)
        state = {**{k:v for k,v in state.items() if not k.startswith('blocks.')},
                 **{f'blocks.{i}.'+k.split('.',2)[2]:v for i,j in enumerate(chosen)
                    for k,v in state.items() if k.startswith(f'blocks.{j}.')}}
    lm = LegalLM(config)
    lm.load_state_dict(state,strict=True)
    del saved,state
    for b,j in zip(lm.blocks,chosen):
        b.attn.global_attn = (j+1)%config['global_every']==0
        b.attn.qkv = SplitQKV(b.attn.qkv,config)
    if method not in ['full','distill']: lm.requires_grad_(False)
    if method=='partial':
        for b in lm.blocks[-6:]: b.requires_grad_(True)
        lm.norm.requires_grad_(True)
    targets = ['q_proj','v_proj','out','up','down']
    if method in ['lora','lorma','ia3']:
        for name,layer in list(lm.named_modules()):
            if not isinstance(layer,nn.Linear): continue
            key = name.split('.')[-1]
            wrapped = None
            if method=='lora' and key in targets: wrapped = LoRALinear(layer,r['rank'],2*r['rank'])
            if method=='lorma' and key in targets: wrapped = LoRMALinear(layer,r['rank'],2*r['rank'])
            if method=='ia3' and key in ['k_proj','v_proj','down']:
                wrapped = IA3Linear(layer,feedforward=key=='down')
            if wrapped is not None:
                parent,key = name.rsplit('.',1)
                setattr(lm.get_submodule(parent),key,wrapped)
    model = CaseModel(lm,method,r,methods[method]['coverage'])
    return model

def case_tokens(row, cache):
    return cache[row['offset']:row['offset']+row['size']]

@torch.inference_mode()
def predict(model, rows, cache):
    model.eval()
    logits = np.empty((len(rows),2),dtype=np.float32)
    for i,row in enumerate(rows):
        with torch.autocast('cuda',dtype=torch.float16):
            z = model(case_tokens(row,cache))
        logits[i] = z.float().cpu().numpy()[0]
    return logits

def probabilities(z):
    z = z-z.max(axis=1,keepdims=True)
    p = np.exp(z)
    return p/p.sum(axis=1,keepdims=True)

def choose_threshold(gold, p):
    thresholds = sorted(np.linspace(.2,.8,61),key=lambda t:abs(t-.5))
    scores = [f1_score(gold,p>=t,labels=[0,1],average='macro',zero_division=0) for t in thresholds]
    k = int(np.argmax(scores))
    return float(thresholds[k]),float(scores[k])

def write_submission(rows, p, threshold, path):
    if len(p)!=len(rows) or not np.isfinite(p).all():
        raise ValueError('Invalid case predictions')
    payload = json.dumps([dict(id=row['id'],pred=int(v>=threshold)) for row,v in zip(rows,p)],
                         ensure_ascii=False,indent=2)
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z: z.writestr('prediction.json',payload)
    return hashlib.sha256(payload.encode()).hexdigest()


## Training and validation

In [7]:
def run_method(gpu, method, c, r, methods, teacher=None):
    torch.set_num_threads(2)
    torch.cuda.set_device(gpu)
    device = torch.device('cuda',gpu)
    torch.manual_seed(r['seed'])
    np.random.seed(r['seed'])
    random.seed(r['seed'])
    root = Path(r['out'])
    folder = root/method
    folder.mkdir(exist_ok=True)
    if (folder/'done.json').exists():
        print(f'GPU {gpu} | {method} already complete; reusing outputs',flush=True)
        return
    with (root/'train_dev.pkl').open('rb') as f: data = cloudpickle.load(f)
    train,dev = data['train'],data['dev']
    caches = {s:np.memmap(root/f'{s}_tokens.bin',dtype=np.int32,mode='r') for s in ['train','dev']}
    gold = np.array([v['y'] for v in dev])
    soft = None
    if method=='distill':
        tm = make_model(teacher,c,r,methods)
        ck = torch.load(root/teacher/'best.pt',map_location='cpu',weights_only=False)
        tm.load_state_dict(ck['weights'],strict=False)
        tm.to(device)
        soft = predict(tm,train,caches['train'])
        np.save(folder/'teacher_logits.npy',soft)
        del tm,ck
        gc.collect()
        torch.cuda.empty_cache()
    model = make_model(method,c,r,methods).to(device)
    params = [(n,p) for n,p in model.named_parameters() if p.requires_grad]
    body = [p for n,p in params if n.startswith('lm.')]
    head = [p for n,p in params if not n.startswith('lm.')]
    lr = methods[method]['lr']
    opt = torch.optim.AdamW([dict(params=body,lr=lr,initial_lr=lr),
                             dict(params=head,lr=3e-4,initial_lr=3e-4)],weight_decay=.01)
    scaler = torch.amp.GradScaler('cuda',init_scale=1024)
    count = np.bincount([v['y'] for v in train],minlength=r['classes'])
    weight = 1/np.sqrt(np.maximum(count,1))
    weight = np.clip(weight/weight.mean(),.25,4.)
    weight = torch.tensor(weight,dtype=torch.float32,device=device)
    nsteps = math.ceil(len(train)/(r['micro']*r['accum']))
    total = methods[method]['epochs']*nsteps
    warmup = max(10,int(.06*total))
    best,stale,step,history = -1.,0,0,[]
    start = time.monotonic()
    total_params = sum(p.numel() for p in model.parameters())
    trained = sum(p.numel() for _,p in params)
    torch.cuda.reset_peak_memory_stats(device)
    print(f'{method} | GPU {gpu} ({torch.cuda.get_device_name(device)}) | model on {next(model.parameters()).device} | {total_params:,} parameters | {trained:,} trainable',flush=True)
    stop = False
    for epoch in range(1,methods[method]['epochs']+1):
        order = np.random.default_rng(r['seed']+epoch).permutation(len(train))
        loss_sum,seen = 0.,0
        for at in range(0,len(order),r['micro']*r['accum']):
            model.train()
            opt.zero_grad(set_to_none=True)
            group = order[at:at+r['micro']*r['accum']]
            factor = min(1.,(step+1)/warmup)*(.1+.9*.5*(1+math.cos(math.pi*min(1.,step/total))))
            for g in opt.param_groups: g['lr'] = g['initial_lr']*factor
            for offset in range(0,len(group),r['micro']):
                ix = group[offset:offset+r['micro']]
                batch = [train[int(i)] for i in ix]
                y = torch.tensor([row['y'] for row in batch],device=device)
                with torch.autocast('cuda',dtype=torch.float16):
                    z = torch.cat([model(case_tokens(row,caches['train'])) for row in batch])
                    ce = (F.cross_entropy(z.float(),y,reduction='none')*weight[y]).mean()
                    loss = ce
                    if soft is not None:
                        target = torch.from_numpy(soft[ix]).to(device)
                        kd = F.kl_div(F.log_softmax(z.float()/2.,-1),
                                      F.softmax(target/2.,-1),reduction='batchmean')*4.
                        loss = .5*ce+.5*kd
                    scaled = loss*(len(ix)/len(group))
                scaler.scale(scaled).backward()
                loss_sum += loss.detach().item()*len(ix)
                seen += len(ix)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for _,p in params],1.)
            scaler.step(opt)
            scaler.update()
            step += 1
            evaluate = step%r['eval_steps']==0 or at+len(group)>=len(order)
            if evaluate:
                logits = predict(model,dev,caches['dev'])
                score = f1_score(gold,logits.argmax(1),labels=list(range(r['classes'])),
                                 average='macro',zero_division=0)
                improved = score>best
                stale = 0 if improved else stale+1
                if improved:
                    best = float(score)
                    weights = {n:p.detach().cpu().float() for n,p in params}
                    torch.save(dict(weights=weights,method=method,config=model.lm.c,
                                    epoch=epoch,step=step,score=best,teacher=teacher),folder/'best.tmp')
                    os.replace(folder/'best.tmp',folder/'best.pt')
                    np.save(folder/'dev_logits.npy',logits)
                row = dict(epoch=epoch,step=step,loss=loss_sum/max(seen,1),macro_f1=float(score))
                history.append(row)
                (folder/'history.json').write_text(json.dumps(history,indent=2))
                print(f'{method} | epoch {epoch} step {step} | loss {row["loss"]:.4f} | dev F1 {score:.4f}',flush=True)
                if stale>=r['patience']:
                    stop=True
                    break
        if stop: break
    ck = torch.load(folder/'best.pt',map_location='cpu',weights_only=False)
    dev_logits = np.load(folder/'dev_logits.npy')
    threshold,calibrated = choose_threshold(gold,probabilities(dev_logits)[:,1])
    pred = (probabilities(dev_logits)[:,1]>=threshold).astype(int)
    report = classification_report(gold,pred,labels=list(range(r['classes'])),
                                   target_names=['0','1'],zero_division=0,output_dict=True)
    (folder/'classification_report.json').write_text(json.dumps(report,indent=2))
    result = dict(method=method,run_id=r['run_id'],macro_f1=best,parameters=total_params,trainable=trained,
                  epoch=ck['epoch'],step=ck['step'],last_step=step,teacher=teacher,
                  stopping_reason='patience' if stop else 'epoch_limit',
                  minutes=(time.monotonic()-start)/60,
                  peak_gb=torch.cuda.max_memory_allocated(device)/2**30)
    result.update(threshold=threshold, calibrated_macro_f1=calibrated,
                  coverage=methods[method]['coverage'])
    (folder/'done.json').write_text(json.dumps(result,indent=2))


## Two-GPU training

In [8]:
def run_jobs(names, worker=run_method):
    pending, running, finished = list(names), {}, set()
    first = ['full','partial','lora','lorma','ia3']
    while pending or running:
        for gpu,(process,path,method) in list(running.items()):
            status = process.poll()
            if status is not None:
                if status != 0:
                    raise subprocess.CalledProcessError(status,process.args)
                path.unlink()
                del running[gpu]
                finished.add(method)
                print(f'GPU {gpu} finished {method}',flush=True)
        for gpu in range(2):
            if gpu in running:
                continue
            ready = [m for m in pending if worker is not run_method or m!='distill' or set(first).issubset(finished)]
            if not ready:
                continue
            method = ready[0]
            teacher = None
            if worker is run_method and method=='distill':
                teacher = max(first,key=lambda m:json.loads((OUT/m/'done.json').read_text())['macro_f1'])
            path = OUT/f'worker_{gpu}.pkl'
            with path.open('wb') as f:
                cloudpickle.dump((worker,(gpu,method,C,R,METHODS,teacher)),f)
            command = "import cloudpickle,sys; f=open(sys.argv[1],'rb'); fn,args=cloudpickle.load(f); f.close(); fn(*args)"
            process = subprocess.Popen([sys.executable,'-u','-c',command,str(path)],env=os.environ.copy())
            running[gpu] = (process,path,method)
            pending.remove(method)
            print(f'GPU {gpu} starting {method} | PID {process.pid}',flush=True)
        if pending and not running:
            raise RuntimeError('Distillation requires the first five methods in this job queue')
        if running:
            time.sleep(.5)
run_jobs(list(METHODS))


GPU 0 starting full | PID 124
GPU 1 starting partial | PID 125
GPU 0 | full already complete; reusing outputs
GPU 1 | partial already complete; reusing outputs
GPU 0 finished full
GPU 1 finished partial
GPU 0 starting lora | PID 142
GPU 1 starting lorma | PID 143
GPU 0 | lora already complete; reusing outputs
GPU 1 | lorma already complete; reusing outputs
GPU 0 finished lora
GPU 1 finished lorma
GPU 0 starting ia3 | PID 160
GPU 0 | ia3 already complete; reusing outputs
GPU 0 finished ia3
GPU 0 starting distill | PID 169
distill | GPU 0 (Tesla T4) | model on cuda:0 | 108,643,523 parameters | 108,643,523 trainable
distill | epoch 1 step 200 | loss 0.4945 | dev F1 0.4129
distill | epoch 1 step 400 | loss 0.4929 | dev F1 0.3442
distill | epoch 1 step 600 | loss 0.4813 | dev F1 0.6067
distill | epoch 1 step 800 | loss 0.4519 | dev F1 0.6187
distill | epoch 1 step 1000 | loss 0.4295 | dev F1 0.6777
distill | epoch 1 step 1200 | loss 0.4142 | dev F1 0.5766
distill | epoch 1 step 1400 | loss 

## Test inference

In [9]:
path = hf_hub_download(REPO, 'Phase_4_CJPE/Test/test.json', repo_type='dataset',
                       revision=revision, token=HF_TOKEN)
docs = read_docs(json.loads(Path(path).read_text()), labelled=False)
DATA['test'] = cache_docs('test',docs)
with (OUT/'test.pkl').open('wb') as f: cloudpickle.dump(DATA['test'],f)
del docs

def run_inference(gpu, method, c, r, methods, teacher=None):
    torch.set_num_threads(2)
    torch.cuda.set_device(gpu)
    device = torch.device('cuda',gpu)
    torch.manual_seed(r['seed'])
    np.random.seed(r['seed'])
    random.seed(r['seed'])
    root = Path(r['out'])
    folder = root/method
    with (root/'test.pkl').open('rb') as f: test = cloudpickle.load(f)
    cache = np.memmap(root/'test_tokens.bin',dtype=np.int32,mode='r')
    model = make_model(method,c,r,methods)
    ck = torch.load(folder/'best.pt',map_location='cpu',weights_only=False)
    model.load_state_dict(ck['weights'],strict=False)
    del ck
    model.to(device)
    torch.cuda.synchronize(device)
    start = time.monotonic()
    logits = predict(model,test,cache)
    torch.cuda.synchronize(device)
    seconds = time.monotonic()-start
    np.save(folder/'test_logits.npy',logits)
    result = json.loads((folder/'done.json').read_text())
    sub = root/'submissions'
    sub.mkdir(exist_ok=True)
    write_submission(test,probabilities(logits)[:,1],result['threshold'],
                     sub/f'submission_P4_A4_{method}.zip')
    result.update(test_seconds=seconds,test_cases_per_second=len(test)/max(seconds,1e-6))
    (folder/'done.json').write_text(json.dumps(result,indent=2))
    print(f'{method} | saved test submission',flush=True)

run_jobs(list(METHODS),worker=run_inference)


Phase_4_CJPE/Test/test.json:   0%|          | 0.00/35.3M [00:00<?, ?B/s]

test: 1,517 cases | 7,847,450 BPE tokens
GPU 0 starting full | PID 188
GPU 1 starting partial | PID 189
partial | saved test submission
GPU 1 finished partial
GPU 1 starting lora | PID 200
full | saved test submission
GPU 0 finished full
GPU 0 starting lorma | PID 206
lora | saved test submission
GPU 1 finished lora
GPU 1 starting ia3 | PID 212
lorma | saved test submission
GPU 0 finished lorma
GPU 0 starting distill | PID 218
distill | saved test submission
GPU 0 finished distill
ia3 | saved test submission
GPU 1 finished ia3


## Submissions

In [10]:
gold = np.array([v['y'] for v in DATA['dev']])
results = [json.loads((OUT/m/'done.json').read_text()) for m in METHODS]
ranked = sorted(results,key=lambda v:v['calibrated_macro_f1'],reverse=True)
members = [v['method'] for v in ranked[:3]]
dev_probs = np.stack([probabilities(np.load(OUT/m/'dev_logits.npy'))[:,1] for m in members])
test_probs = np.stack([probabilities(np.load(OUT/m/'test_logits.npy'))[:,1] for m in members])
weights = np.array([max(v['calibrated_macro_f1'],1e-6) for v in ranked[:3]])
candidates = [np.ones(3)/3, weights/weights.sum()]
choices = [choose_threshold(gold,w@dev_probs) for w in candidates]
k = int(np.argmax([score for threshold,score in choices]))
w, (threshold,score) = candidates[k],choices[k]
ensemble = w@test_probs
selection = dict(members=members,weights=w.tolist(),threshold=threshold,dev_macro_f1=score)
(OUT/'ensemble.json').write_text(json.dumps(selection,indent=2))
SUB = OUT/'submissions'
SUB.mkdir(exist_ok=True)
fingerprints = {}
for row in results+[dict(method='ensemble',threshold=threshold)]:
    method = row['method']
    p = ensemble if method=='ensemble' else probabilities(np.load(OUT/method/'test_logits.npy'))[:,1]
    digest = write_submission(DATA['test'],p,row['threshold'],SUB/f'submission_P4_A4_{method}.zip')
    if digest in fingerprints: print(f'Identical predictions: {method} and {fingerprints[digest]}')
    fingerprints[digest] = method
results.append(dict(method='ensemble',calibrated_macro_f1=score,**selection))
(OUT/'results.json').write_text(json.dumps(results,indent=2))
for v in sorted(results,key=lambda v:v['calibrated_macro_f1'],reverse=True):
    print(f'{v["method"]:10s} dev Macro-F1 {v["calibrated_macro_f1"]:.4f}')
print(f'Seven submission ZIPs: {SUB}')


ensemble   dev Macro-F1 0.7619
lora       dev Macro-F1 0.7504
full       dev Macro-F1 0.7392
lorma      dev Macro-F1 0.7264
partial    dev Macro-F1 0.7193
distill    dev Macro-F1 0.7068
ia3        dev Macro-F1 0.5795
Seven submission ZIPs: /kaggle/working/phase4_a4_scratch_train_only/submissions


In [11]:
with zipfile.ZipFile(OUT/'all_submissions_P4_A4.zip','w',zipfile.ZIP_DEFLATED) as z:
    for method in list(METHODS)+['ensemble']:
        p = SUB/f'submission_P4_A4_{method}.zip'
        z.write(p,p.name)
    for name in ['results.json','ensemble.json','manifest.json']: z.write(OUT/name,name)
print(OUT/'all_submissions_P4_A4.zip')


/kaggle/working/phase4_a4_scratch_train_only/all_submissions_P4_A4.zip


## Outputs

## Adapters